# Bài 07 — Hệ trực chuẩn và ma trận trực giao với NumPy

Tài liệu đi kèm gồm [lý thuyết](notes.md) và [ứng dụng ML](ml-in-practice.md).

## Mục tiêu

- Đọc ma trận Gram để phân biệt trực giao và trực chuẩn.
- Dự đoán kết quả của phép quay, phản xạ và phép chiếu.
- Dựng một basis bằng Gram–Schmidt, kiểm chứng QR và phần dư least squares.
- Kiểm tra hình học của cả batch embedding và tìm truy vấn bị đổi láng giềng khi nén.

Hãy tính ví dụ nhỏ trên giấy trước khi chạy ô tương ứng. Cuối notebook có bảy bài tự làm và một bài tổng hợp.

## 0. Khởi tạo

Các ví dụ chỉ cần NumPy. Ta dùng số thực và so sánh với sai số tuyệt đối nhỏ cho những số có độ lớn như trong bài.

In [1]:
import numpy as np

rng = np.random.default_rng(42)
ATOL = 1e-12

## 1. Đọc từng phần tử của ma trận Gram

Hai cột của $B$ là $(1,1)^T$ và $(2,-2)^T$. Dự đoán hai phần tử trên đường chéo của $B^TB$. Sau khi chuẩn hóa theo cột, những phần tử nào thay đổi?

In [2]:
B = np.array([[1., 2.], [1., -2.]])
lengths = np.linalg.norm(B, axis=0)       # (2,), một norm trên mỗi cột
unit_cols = B / lengths                   # chia mỗi cột cho norm của nó
print("Trước khi chuẩn hóa:\n", B.T @ B)
print("Sau khi chuẩn hóa:\n", unit_cols.T @ unit_cols)
np.testing.assert_allclose(B.T @ B, np.diag([2., 8.]), atol=ATOL)
np.testing.assert_allclose(unit_cols.T @ unit_cols, np.eye(2), atol=ATOL)

Trước khi chuẩn hóa:
 [[2. 0.]
 [0. 8.]]
Sau khi chuẩn hóa:
 [[ 1.00000000e+00 -2.23711432e-17]
 [-2.23711432e-17  1.00000000e+00]]


## 2. Phép quay và phép phản xạ

Tính trước ảnh của $(3,4)^T$ khi quay $90^\circ$ ngược chiều kim đồng hồ và khi phản xạ qua trục đứng. Với cặp vector $x,y$, kiểm tra định thức, tích vô hướng và khoảng cách trong hai trường hợp.

In [3]:
x = np.array([3., 4.])
y = np.array([4., 3.])
rotation = np.array([[0., -1.], [1., 0.]])
reflection = np.diag([-1., 1.])

for name, U in [("Quay", rotation), ("Phản xạ", reflection)]:
    tx, ty = U @ x, U @ y
    print(name, "đưa x đến", tx, "với det =", np.linalg.det(U))
    np.testing.assert_allclose(U.T @ U, np.eye(2), atol=ATOL)
    np.testing.assert_allclose(U.T @ tx, x, atol=ATOL)
    np.testing.assert_allclose(np.linalg.norm(tx), 5., atol=ATOL)
    np.testing.assert_allclose(tx @ ty, x @ y, atol=ATOL)
    np.testing.assert_allclose(np.linalg.norm(tx - ty), np.linalg.norm(x - y), atol=ATOL)

Quay đưa x đến [-4.  3.] với det = 1.0
Phản xạ đưa x đến [-3.  4.] với det = -1.0


## 3. Chiếu lên một hướng bất kỳ

Tìm điểm trên đường $y=x$ gần $(3,1)^T$ nhất. Dùng hai hướng sinh $u$ và $2u$. Thử đoán hệ số của mỗi hướng trước khi so hai chân đường vuông góc.

In [4]:
point = np.array([3., 1.])
for direction in [np.array([1., 1.]), np.array([2., 2.])]:
    alpha = (direction @ point) / (direction @ direction)
    foot = alpha * direction
    residual = point - foot
    print("Hướng", direction, "hệ số", alpha, "hình chiếu", foot)
    np.testing.assert_allclose(foot, [2., 2.], atol=ATOL)
    np.testing.assert_allclose(direction @ residual, 0., atol=ATOL)
    np.testing.assert_allclose(point @ point, foot @ foot + residual @ residual, atol=ATOL)

Hướng [1. 1.] hệ số 2.0 hình chiếu [2. 2.]
Hướng [2. 2.] hệ số 1.0 hình chiếu [2. 2.]


## 4. Dựng hai hướng bằng Gram–Schmidt

Hai cột của $A$ sinh mặt phẳng $x_1=x_2+x_3$. Bắt đầu với hướng cột thứ nhất. Tính phần còn lại của cột thứ hai sau khi trừ hình chiếu lên hướng đầu. Đó là vector nào trước bước chuẩn hóa?

In [5]:
A = np.array([[1., 1.], [1., 0.], [0., 1.]])   # (3, 2), hai cột độc lập
a1, a2 = A[:, 0], A[:, 1]
q1 = a1 / np.linalg.norm(a1)
v2 = a2 - q1 * (q1 @ a2)
q2 = v2 / np.linalg.norm(v2)
Q_hand = np.column_stack((q1, q2))
R_hand = Q_hand.T @ A

np.testing.assert_allclose(v2, [.5, -.5, 1.], atol=ATOL)
np.testing.assert_allclose(Q_hand.T @ Q_hand, np.eye(2), atol=ATOL)
np.testing.assert_allclose(R_hand, np.triu(R_hand), atol=ATOL)
np.testing.assert_allclose(Q_hand @ R_hand, A, atol=ATOL)
print("R theo cách chọn dấu ở bước chuẩn hóa:\n", R_hand)

R theo cách chọn dấu ở bước chuẩn hóa:
 [[1.41421356e+00 7.07106781e-01]
 [1.66533454e-16 1.22474487e+00]]


## 5. Chiếu hai điểm lên cùng mặt phẳng

$(2,-1,3)^T$ thuộc mặt phẳng vì $2=-1+3$. Thêm vector $(1,-1,-1)^T$ vuông góc với hai cột của $A$. Dự đoán chân đường vuông góc của điểm mới. Mỗi **cột** của `points` dưới đây là một điểm, nên cả hai có thể được chiếu cùng lúc.

In [6]:
Q, R = np.linalg.qr(A, mode="reduced")
on_plane = np.array([2., -1., 3.])
normal = np.array([1., -1., -1.])
points = np.column_stack((on_plane, on_plane + normal))  # (3, 2)
coords = Q.T @ points                                    # (2, 2)
feet = Q @ coords                                        # (3, 2)
residuals = points - feet
P = Q @ Q.T                                              # (3, 3)

np.testing.assert_allclose(Q @ R, A, atol=ATOL)
np.testing.assert_allclose(P, Q_hand @ Q_hand.T, atol=ATOL)
np.testing.assert_allclose(P, P.T, atol=ATOL)
np.testing.assert_allclose(P @ P, P, atol=ATOL)
np.testing.assert_allclose(feet, np.column_stack((on_plane, on_plane)), atol=ATOL)
np.testing.assert_allclose(residuals[:, 1], normal, atol=ATOL)
np.testing.assert_allclose(Q.T @ residuals, 0., atol=ATOL)
print("Độ dài hai phần dư:", np.linalg.norm(residuals, axis=0))

Độ dài hai phần dư: [9.15513360e-16 1.73205081e+00]


## 6. Least squares qua hệ tam giác

Thêm nhiễu vào sáu quan sát của một đường thẳng. Với design matrix $X$ có cột đầu toàn 1, $Q^Ty$ đo tọa độ của dự đoán trong basis trực chuẩn. Giải $Rw=Q^Ty$ để tìm intercept và slope theo hai cột của $X$. Liệu phần dư có thể vừa khác zero vừa vuông góc với từng cột của $X$?

In [7]:
t = np.arange(6.)
X = np.column_stack((np.ones_like(t), t))       # (6, 2)
target = 1. + 2. * t + rng.normal(0., .2, size=t.size)
Q_fit, R_fit = np.linalg.qr(X, mode="reduced")
w = np.linalg.solve(R_fit, Q_fit.T @ target)    # (2,)
fitted = X @ w
fit_residual = target - fitted
w_ref, _, rank, _ = np.linalg.lstsq(X, target, rcond=None)

assert rank == 2
np.testing.assert_allclose(w, w_ref, atol=ATOL)
np.testing.assert_allclose(fitted, Q_fit @ (Q_fit.T @ target), atol=ATOL)
np.testing.assert_allclose(X.T @ fit_residual, 0., atol=ATOL)
print("Intercept và slope:", w)
print("Độ dài phần dư:", np.linalg.norm(fit_residual))

Intercept và slope: [1.07452527 1.93955701]
Độ dài phần dư: 0.4752431246781496


## 7. Khi các cột trùng nhau

Nhân đôi cùng một feature để tạo bảng $D$ có rank 1. QR rút gọn trả bao nhiêu cột? Thử chiếu $(0,1,0)^T$ bằng tất cả các cột đó rồi so với hình chiếu lên range thực của $D$. Ví dụ dùng các trục tọa độ để ta thấy ngay hình chiếu đúng bằng tay.

In [8]:
D = np.array([[1., 2.], [0., 0.], [0., 0.]])
probe = np.array([0., 1., 0.])
Q_all, R_all = np.linalg.qr(D, mode="reduced")
left, _, _ = np.linalg.svd(D, full_matrices=False)
rank_D = np.linalg.matrix_rank(D)
Q_range = left[:, :rank_D]
projected = Q_range @ (Q_range.T @ probe)
weights = np.linalg.lstsq(D, probe, rcond=None)[0]

print("Số cột Q từ QR:", Q_all.shape[1], "  Rank của D:", rank_D)
print("Chiếu lên các cột Q của QR:", Q_all @ (Q_all.T @ probe))
print("Chiếu lên range của D:", projected)
np.testing.assert_allclose(projected, np.zeros(3), atol=ATOL)
np.testing.assert_allclose(D @ weights, projected, atol=ATOL)

Số cột Q từ QR: 2   Rank của D: 1
Chiếu lên các cột Q của QR: [0. 1. 0.]
Chiếu lên range của D: [0. 0. 0.]


## Bài tập

Thay mỗi dòng bắt đầu bằng `# TODO:` bằng mã Python của bạn. Chạy ô bài làm rồi chạy ô kiểm tra ngay sau đó. Ô kiểm tra sẽ nhắc tên biến nếu còn thiếu. Mở đáp án thu gọn sau khi đã thử. Các bài dùng dữ liệu riêng để bạn có thể làm lại từng bài sau khi chạy phần khởi tạo và ô dưới đây.

In [9]:
def ready(*names):
    missing = [name for name in names if name not in globals()]
    if missing:
        print("Cần điền TODO cho " + ", ".join(missing))
    return not missing

### Bài 1 — Hai cột đơn vị đã trực chuẩn chưa?

Tính ma trận Gram trước và sau khi chuẩn hóa từng cột của `B1`. Kiểm tra xem phần tử ngoài đường chéo cho phép kết luận gì.

In [10]:
B1 = np.array([[3., 1.], [4., 0.], [0., 0.]])
# TODO: gram1 = ...
# TODO: units1 = ...           # (3, 2)
# TODO: unit_gram1 = ...

In [11]:
if ready('gram1', 'units1', 'unit_gram1'):
    assert gram1.shape == unit_gram1.shape == (2, 2) and units1.shape == (3, 2)
    np.testing.assert_allclose(gram1, [[25., 3.], [3., 1.]], atol=ATOL)
    np.testing.assert_allclose(units1, [[.6, 1.], [.8, 0.], [0., 0.]], atol=ATOL)
    np.testing.assert_allclose(unit_gram1, [[1., .6], [.6, 1.]], atol=ATOL)
    print('Bài 1 đạt.')

Cần điền TODO cho gram1, units1, unit_gram1


<details>
<summary>Đáp án Bài 1</summary>

```python
gram1 = B1.T @ B1
units1 = B1 / np.linalg.norm(B1, axis=0)
unit_gram1 = units1.T @ units1
```

Đường chéo đã bằng 1, nhưng hai cột vẫn có tích vô hướng 0,6.

</details>

### Bài 2 — Định thức và bảo toàn khoảng cách

Với từng ma trận, tính định thức, kiểm tra điều kiện trực giao bằng `np.allclose` và đo bình phương khoảng cách giữa hai điểm sau biến đổi. Trước khi chạy, chỉ ra hai ma trận có cùng định thức mà tác động khác nhau lên khoảng cách.

In [12]:
maps2 = [np.array([[0., -1.], [1., 0.]]), np.diag([-1., 1.]), np.diag([2., .5])]
x2, y2 = np.array([2., 1.]), np.array([1., 1.])
# TODO: dets2 = ...             # ba định thức theo thứ tự
# TODO: orthogonal2 = ...       # ba giá trị boolean
# TODO: distances2 = ...        # ba bình phương khoảng cách

In [13]:
if ready('dets2', 'orthogonal2', 'distances2'):
    np.testing.assert_allclose(dets2, [1., -1., 1.], atol=ATOL)
    np.testing.assert_array_equal(orthogonal2, [True, True, False])
    np.testing.assert_allclose(distances2, [1., 1., 4.], atol=ATOL)
    print('Bài 2 đạt.')

Cần điền TODO cho dets2, orthogonal2, distances2


<details>
<summary>Đáp án Bài 2</summary>

```python
dets2 = [np.linalg.det(M) for M in maps2]
orthogonal2 = [np.allclose(M.T @ M, np.eye(2), atol=ATOL) for M in maps2]
distances2 = [np.linalg.norm(M @ x2 - M @ y2) ** 2 for M in maps2]
```

Ma trận cuối có định thức 1. Hai điểm lệch nhau dọc theo hướng mà ma trận này kéo dài gấp đôi.

</details>

### Bài 3 — Hình chiếu với hướng chưa chuẩn hóa

Tính hệ số, hình chiếu và phần dư của `x3` trên đường thẳng sinh bởi `u3`. Dự đoán dấu hệ số trước khi chạy. Kiểm tra cả độ dài hai thành phần theo Pythagore.

In [14]:
u3 = np.array([2., 0., -2.])
x3 = np.array([1., 2., 3.])
# TODO: alpha3 = ...
# TODO: foot3 = ...
# TODO: residual3 = ...

In [15]:
if ready('alpha3', 'foot3', 'residual3'):
    assert np.shape(alpha3) == () and foot3.shape == residual3.shape == (3,)
    np.testing.assert_allclose(alpha3, -.5, atol=ATOL)
    np.testing.assert_allclose(foot3, [-1., 0., 1.], atol=ATOL)
    np.testing.assert_allclose(foot3 + residual3, x3, atol=ATOL)
    np.testing.assert_allclose(u3 @ residual3, 0., atol=ATOL)
    np.testing.assert_allclose(x3 @ x3, foot3 @ foot3 + residual3 @ residual3, atol=ATOL)
    print('Bài 3 đạt.')

Cần điền TODO cho alpha3, foot3, residual3


<details>
<summary>Đáp án Bài 3</summary>

```python
alpha3 = (u3 @ x3) / (u3 @ u3)
foot3 = alpha3 * u3
residual3 = x3 - foot3
```

Hệ số âm vì tích vô hướng âm. Chân đường vuông góc nằm trên tia đối của hướng đã chọn.

</details>

### Bài 4 — Tự cài Gram–Schmidt cho hai cột

Viết `gs_two(A, rtol=1e-10)` nhận một ma trận thực có hai cột. Trả về hai cột trực chuẩn theo thứ tự đầu vào. Nếu cột đầu bằng zero hoặc độ dài còn lại của cột thứ hai không vượt `rtol` nhân với chuẩn cột thứ hai, hãy báo `ValueError`. So kết quả với QR qua ma trận chiếu.

In [16]:
A4 = np.array([[1., 1.], [1., 0.], [1., 2.]])
# TODO: def gs_two(A, rtol=1e-10):
#           ...

In [17]:
if ready('gs_two'):
    basis4 = gs_two(A4)
    ref4, _ = np.linalg.qr(A4, mode="reduced")
    assert basis4.shape == (3, 2)
    np.testing.assert_allclose(basis4[:, 0], A4[:, 0] / np.linalg.norm(A4[:, 0]), atol=ATOL)
    np.testing.assert_allclose(basis4.T @ basis4, np.eye(2), atol=ATOL)
    np.testing.assert_allclose(basis4 @ basis4.T, ref4 @ ref4.T, atol=ATOL)
    for bad in [np.zeros((3, 2)), np.array([[1., 2.], [0., 0.]]), np.array([[1., 1.], [0., 1e-12]])]:
        with np.testing.assert_raises(ValueError):
            gs_two(bad)
    print('Bài 4 đạt.')

Cần điền TODO cho gs_two


<details>
<summary>Đáp án Bài 4</summary>

```python
def gs_two(A, rtol=1e-10):
    first, second = A[:, 0], A[:, 1]
    length = np.linalg.norm(first)
    if length == 0.:
        raise ValueError("Cột đầu có norm bằng 0")
    q1 = first / length
    remainder = second - q1 * (q1 @ second)
    remaining_length = np.linalg.norm(remainder)
    if remaining_length <= rtol * np.linalg.norm(second):
        raise ValueError("Cột thứ hai không thêm hướng độc lập ở ngưỡng đã chọn")
    return np.column_stack((q1, remainder / remaining_length))
```

Chuẩn còn lại đo phần của cột thứ hai vuông góc với cột đầu. Ngưỡng tương đối ở đây cho phép nhận diện hai hướng gần song song.

</details>

### Bài 5 — Sửa phép chiếu lên hai hướng xiên

Đoạn `B5 @ (B5.T @ x5)` chạy được. Hãy kiểm tra nó có thực sự cho chân đường vuông góc không. Lấy basis bằng QR, dựng ma trận chiếu và tính lại kết quả. Vì sao hai thành phần cuối của điểm sau chiếu phải bằng nhau?

In [18]:
B5 = np.array([[2., 1.], [0., 1.], [0., 1.]])
x5 = np.array([2., 1., -3.])
wrong5 = B5 @ (B5.T @ x5)
# TODO: Q5, R5 = ...
# TODO: P5 = ...
# TODO: foot5 = ...
# TODO: wrong_residual_dots5 = ...     # B5.T nhân với phần dư của cách ban đầu

In [19]:
if ready('Q5', 'R5', 'P5', 'foot5', 'wrong_residual_dots5'):
    assert Q5.shape == (3, 2) and P5.shape == (3, 3) and foot5.shape == (3,)
    np.testing.assert_allclose(Q5 @ R5, B5, atol=ATOL)
    np.testing.assert_allclose(Q5.T @ Q5, np.eye(2), atol=ATOL)
    np.testing.assert_allclose(P5, Q5 @ Q5.T, atol=ATOL)
    np.testing.assert_allclose(P5, P5.T, atol=ATOL)
    np.testing.assert_allclose(P5 @ P5, P5, atol=ATOL)
    np.testing.assert_allclose(foot5, [2., -1., -1.], atol=ATOL)
    np.testing.assert_allclose(B5.T @ (x5 - foot5), 0., atol=ATOL)
    np.testing.assert_allclose(wrong_residual_dots5, [-12., -8.], atol=ATOL)
    print('Bài 5 đạt.')

Cần điền TODO cho Q5, R5, P5, foot5, wrong_residual_dots5


<details>
<summary>Đáp án Bài 5</summary>

```python
Q5, R5 = np.linalg.qr(B5, mode="reduced")
P5 = Q5 @ Q5.T
foot5 = P5 @ x5
wrong_residual_dots5 = B5.T @ (x5 - wrong5)
```

Các hàng cuối của B5 giống nhau, nên mọi vector trong range có hai thành phần cuối bằng nhau. Phép chiếu giữ tọa độ đầu và đưa hai tọa độ cuối về trung bình của chúng.

</details>

### Bài 6 — Fit một đường thẳng bằng QR

Dùng QR rút gọn để fit $y\simeq w_0+w_1 t$. Tính trọng số bằng `solve`, phần dư trên train và dự đoán tại $t=4$. Có thể dùng `lstsq` để đối chiếu sau khi làm. Cột intercept khiến tổng phần dư trên train có tính chất gì?

In [20]:
t6 = np.arange(4.)
X6 = np.column_stack((np.ones_like(t6), t6))
y6 = np.array([1., 2., 2., 5.])
# TODO: Q6, R6 = ...
# TODO: w6 = ...
# TODO: residual6 = ...
# TODO: prediction6 = ...       # scalar, một mẫu mới

In [21]:
if ready('Q6', 'R6', 'w6', 'residual6', 'prediction6'):
    assert Q6.shape == (4, 2) and R6.shape == (2, 2)
    assert w6.shape == (2,) and residual6.shape == (4,) and np.shape(prediction6) == ()
    np.testing.assert_allclose(Q6 @ R6, X6, atol=ATOL)
    np.testing.assert_allclose(Q6.T @ Q6, np.eye(2), atol=ATOL)
    np.testing.assert_allclose(w6, np.linalg.lstsq(X6, y6, rcond=None)[0], atol=ATOL)
    np.testing.assert_allclose(residual6, y6 - X6 @ w6, atol=ATOL)
    np.testing.assert_allclose(X6.T @ residual6, 0., atol=ATOL)
    np.testing.assert_allclose(prediction6, 5.5, atol=ATOL)
    print('Bài 6 đạt.')

Cần điền TODO cho Q6, R6, w6, residual6, prediction6


<details>
<summary>Đáp án Bài 6</summary>

```python
Q6, R6 = np.linalg.qr(X6, mode="reduced")
w6 = np.linalg.solve(R6, Q6.T @ y6)
residual6 = y6 - X6 @ w6
prediction6 = np.array([1., 4.]) @ w6
```

Các hệ số là 0,7 và 1,2. Phần dư vuông góc với cột toàn 1, nên tổng các phần dư trên train bằng zero.

</details>

### Bài 7 — Phát hiện lỗi quay embedding theo batch

Bộ mã hóa truy vấn dùng `U7 @ x7`. Kho ảnh lưu từng mẫu theo hàng. Tính lại embedding của cả kho, khoảng cách bình phương đến từng ảnh và chỉ số ảnh gần nhất. Sau đó tính các khoảng cách khi truy vấn đã quay nhưng kho vẫn ở hệ cũ. Trong trường hợp này, chỉ kiểm tra shape có phát hiện lỗi không?

In [22]:
items7 = np.array([[4., 3.], [0., 5.], [-3., 4.]])
x7 = np.array([3., 4.])
U7 = np.array([[0., -1.], [1., 0.]])
query7 = U7 @ x7
# TODO: rotated7 = ...
# TODO: distances7 = ...
# TODO: best7 = ...
# TODO: mixed7 = ...

In [23]:
if ready('rotated7', 'distances7', 'best7', 'mixed7'):
    assert rotated7.shape == items7.shape and distances7.shape == mixed7.shape == (3,)
    np.testing.assert_allclose(rotated7, np.stack([U7 @ row for row in items7]), atol=ATOL)
    np.testing.assert_allclose(distances7, [2., 10., 36.], atol=ATOL)
    assert best7 == 0
    np.testing.assert_allclose(mixed7, [64., 20., 2.], atol=ATOL)
    assert np.argmin(mixed7) == 2
    print('Bài 7 đạt.')

Cần điền TODO cho rotated7, distances7, best7, mixed7


<details>
<summary>Đáp án Bài 7</summary>

```python
rotated7 = items7 @ U7.T
distances7 = np.sum((rotated7 - query7) ** 2, axis=1)
best7 = np.argmin(distances7)
mixed7 = np.sum((items7 - query7) ** 2, axis=1)
```

Hai bảng có cùng shape. Một cặp truy vấn–ảnh với khoảng cách đã biết giúp phát hiện hai phía đang dùng hai hệ khác nhau.

</details>

## Bài tổng hợp — Tìm ảnh gần nhất sau khi quay và nén

Ba ảnh A, B, C và hai truy vấn có embedding ba chiều. Ô dưới cung cấp một phép quay và hai hướng độc lập cần giữ khi nén. Mỗi hàng trong bảng dữ liệu là một embedding.

1. Viết `pair_dist2(queries, items)` trả về bình phương khoảng cách của **mọi cặp**, shape `(số truy vấn, số ảnh)`. Đo khoảng cách trong không gian gốc và sau khi quay đồng bộ.
2. Từ hai cột của `directions`, dùng QR để dựng basis trực chuẩn của mặt phẳng. Mã hóa cả hai bảng bằng hai tọa độ, rồi tính khoảng cách trong không gian nén.
3. Tìm ảnh top-1 trước và sau nén cho từng truy vấn. Tính tỷ lệ giữ nguyên top-1. Giải thích vì sao ảnh C có thể trùng với truy vấn đầu sau nén.
4. Dựng phần của từng embedding bị bỏ khi chiếu. Từ các phần dư này, đo phần khoảng cách bình phương bị mất ở mỗi cặp. Kiểm tra đẳng thức Pythagore trên cả bảng.

In [24]:
items = np.array([[1.2, 1.2, 1.], [0., 0., 3.], [3., -1., 1.]])  # A, B, C
queries = np.array([[1., 1., 1.], [0., 0., 2.75]])
turn = np.array([[0., 0., 1.], [1., 0., 0.], [0., 1., 0.]])
directions = np.array([[1., 1.], [1., 1.], [0., 2.]])   # (3, 2)
# TODO: def pair_dist2(queries, items):
#           ...
# TODO: original_d2 = ...
# TODO: rotated_d2 = ...
# TODO: basis, _ = ...
# TODO: item_codes = ...
# TODO: query_codes = ...
# TODO: compressed_d2 = ...
# TODO: top_before = ...
# TODO: top_after = ...
# TODO: agreement = ...
# TODO: lost_d2 = ...      # tính từ phần dư tái tạo của truy vấn và kho

In [25]:
if ready('pair_dist2', 'original_d2', 'rotated_d2', 'basis', 'item_codes', 'query_codes', 'compressed_d2', 'top_before', 'top_after', 'agreement', 'lost_d2'):
    np.testing.assert_allclose(pair_dist2(np.array([[1., 2.]]), np.array([[1., 0.], [0., 2.]])), [[4., 1.]])
    assert original_d2.shape == rotated_d2.shape == compressed_d2.shape == lost_d2.shape == (2, 3)
    assert basis.shape == (3, 2) and item_codes.shape == (3, 2) and query_codes.shape == (2, 2)
    np.testing.assert_allclose(original_d2, [[.08, 6., 8.], [5.9425, .0625, 13.0625]], atol=ATOL)
    np.testing.assert_allclose(rotated_d2, original_d2, atol=ATOL)
    np.testing.assert_allclose(basis.T @ basis, np.eye(2), atol=ATOL)
    np.testing.assert_allclose(basis @ (basis.T @ directions), directions, atol=ATOL)
    np.testing.assert_allclose(item_codes, items @ basis, atol=ATOL)
    np.testing.assert_allclose(query_codes, queries @ basis, atol=ATOL)
    np.testing.assert_allclose(lost_d2, [[0., 0., 8.], [0., 0., 8.]], atol=ATOL)
    np.testing.assert_allclose(original_d2, compressed_d2 + lost_d2, atol=ATOL)
    np.testing.assert_array_equal(top_before, [0, 1])
    np.testing.assert_array_equal(top_after, [2, 1])
    np.testing.assert_allclose(agreement, .5, atol=ATOL)
    print('Bài tổng hợp đạt.')

Cần điền TODO cho pair_dist2, original_d2, rotated_d2, basis, item_codes, query_codes, compressed_d2, top_before, top_after, agreement, lost_d2


<details>
<summary>Đáp án Bài tổng hợp</summary>

```python
def pair_dist2(queries, items):
    diff = queries[:, None, :] - items[None, :, :]
    return np.sum(diff ** 2, axis=2)

original_d2 = pair_dist2(queries, items)
rotated_d2 = pair_dist2(queries @ turn.T, items @ turn.T)
basis, _ = np.linalg.qr(directions, mode="reduced")
item_codes = items @ basis
query_codes = queries @ basis
compressed_d2 = pair_dist2(query_codes, item_codes)
top_before = np.argmin(original_d2, axis=1)
top_after = np.argmin(compressed_d2, axis=1)
agreement = np.mean(top_before == top_after)

item_residuals = items - item_codes @ basis.T
query_residuals = queries - query_codes @ basis.T
lost_d2 = pair_dist2(query_residuals, item_residuals)
```

Hiệu giữa ảnh C và truy vấn đầu là $(2,-2,0)^T$, vuông góc với cả hai hướng giữ lại. Bình phương khoảng cách 8 mất hoàn toàn sau nén. Truy vấn thứ hai vẫn chọn B, nên một trong hai truy vấn giữ nguyên láng giềng.

</details>

## Tự kiểm tra nhanh

1. Nếu hai vector đều có norm 1, ma trận Gram có thể có phần tử ngoài đường chéo bằng $-1$ không? Cho một ví dụ.
2. Trong bài 3, thay $u$ bằng $-3u$ ảnh hưởng thế nào đến hệ số và hình chiếu?
3. Vì sao `Q.T @ Q` và `Q @ Q.T` ở phần 5 có rank bằng nhau dù shape khác nhau?
4. Nếu $R$ từ QR có đường chéo âm ở một vị trí, cần kiểm tra những gì để xác nhận phân rã đúng?
5. Trong bài tổng hợp, giải thích vì sao mọi khoảng cách đều có thể giảm hoặc giữ nguyên mà top-1 vẫn thay đổi.

## Tổng kết

- Ma trận Gram cho biết độ dài và góc giữa các hướng. Với cột trực chuẩn, $Q^TQ=I$.
- Hình chiếu là $Q(Q^Tx)$. Phần dư nằm trong kernel của $Q^T$. Bình phương chuẩn của nó bằng phần chênh lệch giữa bình phương chuẩn của đầu vào và của hình chiếu.
- Gram–Schmidt cho thấy từng bước hình thành basis. QR cho phép chuyển bài toán least squares full column rank thành một hệ tam giác.
- Trong tìm kiếm embedding, phép quay đồng bộ giữ khoảng cách của từng cặp. Khi giảm chiều, hãy xét cả phần dư lẫn kết quả truy xuất.

Trước khi sang Bài 08, thử viết hai công thức riêng cho việc quay một vector và việc đọc tọa độ của vector ấy theo một basis trực chuẩn.